# BBO Capstone — Query 3 Submission
**Imperial College London · Executive Master in ML/AI**  
**Gian Franco Cattaneo · COO, SALOV S.p.A.**

---

## Context
This notebook documents the data analysis, decision rationale, and final query vector for Round 3 of the Black-Box Optimisation capstone.  
The core objective is to use prior observations (Rounds 1 and 2) to refine the search strategy, applying SVM-informed reasoning to guide exploration/exploitation trade-offs across 8 unknown functions of varying dimensionality.

All inputs are normalised to [0, 1]. Outputs are black-box — function forms are unknown.

## 1. Load Prior Observations

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

# ── Round 1 inputs ──────────────────────────────────────────────────────────
inputs_round1 = [
    np.array([0.034388, 0.909319]),                                        # f1 (d=2)
    np.array([0.695196, 0.39597]),                                         # f2 (d=2)
    np.array([0.548145, 0.174647, 0.303245]),                              # f3 (d=3)
    np.array([0.440429, 0.425456, 0.378357, 0.397088]),                    # f4 (d=4)
    np.array([0.0,      0.675974, 0.999999, 0.999999]),                    # f5 (d=4)
    np.array([0.464677, 0.24211,  0.574863, 0.999999, 0.0]),               # f6 (d=5)
    np.array([0.0,      0.241713, 0.327655, 0.218095, 0.375335, 0.747501]),# f7 (d=6)
    np.array([0.064016, 0.008062, 0.123268, 0.0, 0.999999,
               0.381742, 0.031402, 0.80601])                               # f8 (d=8)
]

# ── Round 1 outputs ─────────────────────────────────────────────────────────
outputs_round1 = [
    np.float64(-2.4674747069022486e-270),  # f1
    np.float64(0.7237404632835625),         # f2
    np.float64(-0.08911956876452833),       # f3
    np.float64(0.25957575200735095),        # f4
    np.float64(2105.928152398213),          # f5  ★ dominant
    np.float64(-0.5507747202906804),        # f6
    np.float64(2.207308607344047),          # f7
    np.float64(9.8595486103895)             # f8
]

# ── Round 2 inputs ──────────────────────────────────────────────────────────
inputs_round2 = [
    np.array([0.999999, 0.999999]),                                        # f1
    np.array([0.698486, 0.0]),                                             # f2
    np.array([0.850892, 0.035316, 0.936193]),                              # f3
    np.array([0.999999, 0.0,      0.0,      0.365908]),                    # f4  ← boundary
    np.array([0.0,      0.0,      0.999999, 0.999999]),                    # f5
    np.array([0.142733, 0.321812, 0.416485, 0.999999, 0.304415]),          # f6
    np.array([0.0,      0.302741, 0.0,      0.187177, 0.0,      0.167182]),# f7  ← x3,x5=0
    np.array([0.096074, 0.0,      0.581701, 0.0,      0.999999,
               0.38389,  0.202189, 0.999999])                              # f8
]

# ── Round 2 outputs ─────────────────────────────────────────────────────────
outputs_round2 = [
    np.float64(1.517648729565899e-192),     # f1
    np.float64(0.5297658866453171),          # f2
    np.float64(-0.23982430098711077),        # f3
    np.float64(-27.859767965401783),         # f4  ← catastrophic
    np.float64(1616.625747348229),           # f5
    np.float64(-1.0045153236844038),         # f6
    np.float64(0.050978228653516464),        # f7
    np.float64(9.2933769573024)              # f8
]

print("Data loaded: 8 functions, 2 rounds each")
print(f"Total labelled observations: {len(outputs_round1) + len(outputs_round2)}")

## 2. Cross-Round Delta Analysis

In [ ]:
fn_labels = [f'f{i+1}' for i in range(8)]
dims       = [2, 2, 3, 4, 4, 5, 6, 8]

deltas   = [r2 - r1 for r1, r2 in zip(outputs_round1, outputs_round2)]
pct_chg  = []
for r1, d in zip(outputs_round1, deltas):
    if abs(r1) > 1e-50:
        pct_chg.append(d / abs(r1) * 100)
    else:
        pct_chg.append(float('nan'))

df = pd.DataFrame({
    'Function':  fn_labels,
    'Dim':       dims,
    'R1_output': [round(v, 4) for v in outputs_round1],
    'R2_output': [round(v, 4) for v in outputs_round2],
    'Delta':     [round(d, 4) for d in deltas],
    'Pct_change': [round(p, 1) if not np.isnan(p) else 'N/A' for p in pct_chg]
})

print(df.to_string(index=False))
print(f"\nFunctions with R1 > R2 (deteriorated): {sum(1 for d in deltas if d < 0)} / 8")
print(f"Best observed (R1 f5): {max(outputs_round1):.2f}")
print(f"Worst observed (R2 f4): {min(outputs_round2):.4f}")

## 3. Visualisation — Output Landscape Across Rounds

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={'width_ratios': [2, 1]})

# --- Left: grouped bar chart (exclude f5 in main chart; too large)
ax = axes[0]
fn_show = [f for f in fn_labels if f != 'f5']
idx_show = [fn_labels.index(f) for f in fn_show]

r1_vals = [outputs_round1[i] for i in idx_show]
r2_vals = [outputs_round2[i] for i in idx_show]

x = np.arange(len(fn_show))
w = 0.35
bars1 = ax.bar(x - w/2, r1_vals, w, label='Round 1', color='#185FA5', alpha=0.85)
bars2 = ax.bar(x + w/2, r2_vals, w, label='Round 2', color='#0F6E56', alpha=0.85)

ax.axhline(0, color='#888', linewidth=0.7, linestyle='--')
ax.set_xticks(x)
ax.set_xticklabels(fn_show, fontsize=11)
ax.set_ylabel('Output value', fontsize=11)
ax.set_title('Output comparison R1 vs R2  (f5 excluded — scale 10³)', fontsize=12, pad=10)
ax.legend(fontsize=10)
ax.spines[['top', 'right']].set_visible(False)
ax.tick_params(labelsize=10)

# Annotate catastrophic f4 R2
ax.annotate('f4 R2: −27.86\n(boundary values)', xy=(fn_show.index('f4') + w/2, -27.86),
            xytext=(fn_show.index('f4') + 1.2, -22),
            fontsize=8.5, color='#A32D2D',
            arrowprops=dict(arrowstyle='->', color='#A32D2D', lw=1))

# --- Right: f5 standalone
ax2 = axes[1]
f5_vals = [outputs_round1[4], outputs_round2[4]]
bars3 = ax2.bar(['Round 1', 'Round 2'], f5_vals,
                color=['#BA7517', '#854F0B'], alpha=0.88, width=0.5)
ax2.set_title('f5 ★  (dominant function)', fontsize=12, pad=10)
ax2.set_ylabel('Output value', fontsize=11)
ax2.spines[['top', 'right']].set_visible(False)
ax2.tick_params(labelsize=10)
for bar, val in zip(bars3, f5_vals):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
             f'{val:,.0f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax2.annotate('Δ = −489 units\n(x₂: 0.676 → 0.0)', xy=(1, 1616),
             xytext=(0.5, 1200), fontsize=8.5, color='#A32D2D',
             arrowprops=dict(arrowstyle='->', color='#A32D2D', lw=1))

plt.tight_layout()
plt.suptitle('BBO Capstone — Output Landscape (Rounds 1 & 2)', y=1.02,
             fontsize=13, fontweight='bold')
plt.savefig('bbo_output_landscape.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved to bbo_output_landscape.png")

## 4. SVM-Informed Region Classification

In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# ────────────────────────────────────────────────────────────────────────────
# Demonstrate SVM classification for f5 (d=4) as a worked example.
# Label: 1 = above-median output (HIGH), 0 = below-median (LOW).
# For a real BBO pipeline this would be applied per-function with more data.
# ────────────────────────────────────────────────────────────────────────────

# f5 — two observations (x, y)
X_f5 = np.array([
    [0.0, 0.675974, 0.999999, 0.999999],   # R1 → 2105.93
    [0.0, 0.0,      0.999999, 0.999999],   # R2 → 1616.63
])
y_f5 = np.array([1, 0])  # R1 = HIGH, R2 = LOW (below median of 1861)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_f5)

# Soft-margin SVM (C=1.0) — necessary given potential outliers
svm = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True)
svm.fit(X_scaled, y_f5)

# Query 3 candidate for f5: x2 pushed to 0.85
q3_f5 = np.array([[0.0, 0.85, 0.999999, 0.999999]])
q3_scaled = scaler.transform(q3_f5)
pred_class = svm.predict(q3_scaled)[0]
pred_prob  = svm.predict_proba(q3_scaled)[0]

print("SVM CLASSIFICATION — f5 (d=4)")
print("=" * 40)
print(f"Training points: 2 (R1 = HIGH, R2 = LOW)")
print(f"Kernel: RBF  |  C: 1.0  |  gamma: scale")
print()
print(f"Query 3 candidate: {q3_f5[0]}")
print(f"  Predicted class : {'HIGH ✓' if pred_class == 1 else 'LOW ✗'}")
print(f"  P(HIGH)         : {pred_prob[1]:.3f}")
print(f"  P(LOW)          : {pred_prob[0]:.3f}")
print()
print("Note: With only 2 training points the SVM is indicative, not definitive.")
print("Confidence will increase as observations accumulate across queries.")

## 5. Dimensional Sensitivity Analysis

In [ ]:
# For each function, compute the absolute change in each dimension between R1 and R2
# and cross-reference with the output delta.
# A large input change paired with large output change flags a sensitive dimension.

print("DIMENSIONAL SENSITIVITY ANALYSIS")
print("=" * 55)
print(f"{'Fn':>4}  {'Dim':>3}  {'|Δx|':>8}  {'Δy':>12}  {'Sensitivity flag'}")
print("-" * 55)

for i, (r1_in, r2_in, r1_out, r2_out, label) in enumerate(
    zip(inputs_round1, inputs_round2, outputs_round1, outputs_round2, fn_labels)
):
    delta_x = np.abs(r2_in - r1_in)
    delta_y = r2_out - r1_out
    active  = []

    # Flag: dimension where |Δx| > 0.3 AND |Δy| > 0.5
    for j, dx in enumerate(delta_x):
        if dx > 0.3 and abs(delta_y) > 0.5:
            active.append(f'x{j+1}(Δ={dx:.2f})')

    flag = ', '.join(active) if active else '—'
    print(f"{label:>4}  {dims[i]:>3}  {np.sum(delta_x):>8.4f}  {delta_y:>12.4f}  {flag}")

print()
print("Interpretation:")
print("  f5  → x2 moved 0.676 units, output dropped 489. Confirmed active.")
print("  f7  → x3 and x5 zeroed simultaneously, output collapsed 97.7%.")
print("  f4  → x1 jumped to boundary (0.999999), output catastrophically negative.")
print("  f8  → x3 increased 0.46 units; moderate output decline observed.")

## 6. Query 3 — Final Input Vectors

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  QUERY 3 — FINAL SUBMISSION VECTORS
# ════════════════════════════════════════════════════════════════════════════

query_3_inputs = [
    np.array([0.250000, 0.250000]),                                               # f1  explore: lower-left quadrant
    np.array([0.695000, 0.396000]),                                               # f2  exploit: near R1 best
    np.array([0.300000, 0.500000, 0.700000]),                                     # f3  explore: diagonal sweep
    np.array([0.440000, 0.425000, 0.378000, 0.397000]),                           # f4  exploit: interior-constrained
    np.array([0.000000, 0.850000, 0.999999, 0.999999]),                           # f5  push:    x2 → 0.85
    np.array([0.500000, 0.500000, 0.500000, 0.500000, 0.500000]),                 # f6  explore: centroid baseline
    np.array([0.000000, 0.242000, 0.328000, 0.218000, 0.375000, 0.748000]),       # f7  exploit: near R1 exactly
    np.array([0.064000, 0.008000, 0.120000, 0.000000, 0.999999,
               0.382000, 0.031000, 0.806000])                                     # f8  exploit: R1 with x3 reduced
]

# Strategy summary per function
strategies = [
    'EXPLORE  — both corners gave ≈0; test lower-left quadrant',
    'EXPLOIT  — x2 drop to 0 cost 0.19 units; return to x2=0.396',
    'EXPLORE  — persistently negative; sweep new 3D diagonal',
    'EXPLOIT  — boundary values catastrophic; stay in [0.38,0.44]',
    'PUSH     — test monotonic x2 hypothesis: x2=0.85 vs R1 x2=0.676',
    'EXPLORE  — centroid as SVM baseline for future labelling',
    'EXPLOIT  — x3,x5=0 in R2 caused 97.7% collapse; restore R1',
    'EXPLOIT  — x5=1,x4=0 fixed; reduce x3 to R1 level (0.12)',
]

print("QUERY 3 — SUBMISSION VECTORS")
print("=" * 72)
for fn, vec, strat, dim in zip(fn_labels, query_3_inputs, strategies, dims):
    print(f"  {fn} (d={dim}): {strat}")
    print(f"       → {np.round(vec, 6).tolist()}")
    print()

print("Copy-paste ready format:")
print("─" * 72)
print("[")
for i, (fn, vec) in enumerate(zip(fn_labels, query_3_inputs)):
    suffix = ',  # ' + fn + f' (d={dims[i]})'
    print(f"  array({np.round(vec,6).tolist()}){suffix}")
print("]")

## 7. Expected Output Hypotheses

In [ ]:
# Document testable hypotheses for each function in Query 3.
# These will be validated when results are returned.

hypotheses = {
    'f1': (
        'Output remains ≈ 0',
        'Function is flat across the [0,1]² domain; optimum not yet localised'
    ),
    'f2': (
        'Output ≥ 0.70 (near R1 level)',
        'Near-identical input to R1; confirms that x2≈0.40 is important'
    ),
    'f3': (
        'Output potentially positive',
        'Diagonal sweep explores untested region; may reveal positive subspace'
    ),
    'f4': (
        'Output ≈ +0.25 (near R1 level)',
        'Interior-constrained inputs should recover R1 performance'
    ),
    'f5': (
        'Output > 2105.93 (exceeds R1 best)',
        'If x2 drives output monotonically, x2=0.85 > 0.676 should exceed R1'
    ),
    'f6': (
        'Output near 0 or slightly negative',
        'Centroid provides a neutral reference; any positive output is informative'
    ),
    'f7': (
        'Output ≈ 2.20 (near R1 level)',
        'Near-exact R1 replication; confirms x3,x5 importance'
    ),
    'f8': (
        'Output > 9.29 (better than R2)',
        'Reducing x3 to 0.12 should recover the R1 advantage'
    ),
}

print("QUERY 3 — TESTABLE HYPOTHESES")
print("=" * 60)
for fn, (pred, reason) in hypotheses.items():
    print(f"  {fn}: {pred}")
    print(f"     Rationale: {reason}")
    print()

print("On receipt of Query 3 results, update each hypothesis above.")
print("Confirmed hypotheses become priors for Query 4 GP/SVM models.")

---
## Summary

| Function | Dim | Strategy | Key Rationale |
|----------|-----|----------|---------------|
| f1 | 2 | Explore | Zero output at both tested corners |
| f2 | 2 | Exploit | R1 was better; x₂ drop caused decline |
| f3 | 3 | Explore | Persistently negative; new region needed |
| f4 | 4 | Exploit | Boundary inputs catastrophic; stay interior |
| f5 | 4 | Push x₂ | Monotonic hypothesis: x₂=0.85 > R1 x₂=0.676 |
| f6 | 5 | Explore | Centroid baseline for future SVM labelling |
| f7 | 6 | Exploit | Zeroing x₃,x₅ caused 97.7% collapse in R2 |
| f8 | 8 | Exploit | x₅=1, x₄=0 fixed; reduce x₃ to recover R1 |

**SVM integration (future rounds):**  
Accumulate 3+ observations per function → apply binary SVM (median threshold) → use decision boundary as pre-filter for GP-BO acquisition function candidates.  
Recommended kernel: RBF for non-linear surfaces (f5, f8); soft-margin C∈[0.1, 10] to handle f4-style outliers.  
Per-function normalisation mandatory before joint modelling (4 orders of magnitude scale range across functions).